# Automated Web Security Header & CORS Checker

**Project type:** Python cybersecurity mini-project  
**Purpose:** Inspect common HTTP security headers and make a basic observation of a site's CORS response.  
**Authorization:** Run only against websites you own or have explicit permission to test.

## How it works
1. Accepts a full HTTP/HTTPS URL.
2. Sends a normal GET request and follows redirects.
3. Checks six commonly recommended security headers.
4. Sends a second GET request with a harmless test Origin to observe CORS headers.
5. Prints the findings and saves them to `scan_results.json`.

**Limitations:** This is an educational checker, not a penetration-testing tool. Missing headers are not automatically exploitable vulnerabilities, and CORS requires context-specific review.

In [ ]:
"""
Automated Web Security Header & CORS Checker
Use only against websites you own or are explicitly authorized to test.
This tool sends ordinary HTTP GET requests to a single URL. It does not exploit
vulnerabilities, crawl pages, or attempt authentication bypass.
"""
from urllib.parse import urlparse
import requests
import json
from datetime import datetime

SECURITY_HEADERS = {
    "Strict-Transport-Security": "Helps enforce HTTPS connections.",
    "Content-Security-Policy": "Helps reduce the impact of content injection.",
    "X-Content-Type-Options": "Helps prevent MIME-type sniffing.",
    "X-Frame-Options": "Helps reduce clickjacking risk (CSP frame-ancestors is a modern alternative).",
    "Referrer-Policy": "Controls how much referrer information is sent.",
    "Permissions-Policy": "Restricts access to browser features."
}

def validate_url(url):
    """Accept only absolute HTTP/HTTPS URLs with a hostname."""
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Enter a full URL beginning with http:// or https://")
    return url.strip()

def check_website(url, timeout=10):
    """
    Check security response headers and a basic CORS response.
    Only run this on a site you own or have permission to assess.
    """
    url = validate_url(url)
    results = {
        "target": url,
        "checked_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "http_status": None,
        "final_url": None,
        "security_headers": {},
        "cors": {},
        "errors": []
    }

    try:
        response = requests.get(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={"User-Agent": "Educational-WebSecurity-Header-Checker/1.0"},
        )
        results["http_status"] = response.status_code
        results["final_url"] = response.url
        headers = response.headers

        for header, purpose in SECURITY_HEADERS.items():
            value = headers.get(header)
            results["security_headers"][header] = {
                "present": value is not None,
                "value": value or "",
                "purpose": purpose
            }

        # A harmless test Origin is used to observe whether the server reflects it.
        # This is a basic indicator only, not a complete CORS audit.
        try:
            cors_response = requests.get(
                url,
                timeout=timeout,
                allow_redirects=True,
                headers={
                    "User-Agent": "Educational-WebSecurity-Header-Checker/1.0",
                    "Origin": "https://security-check.example.invalid"
                },
            )
            acao = cors_response.headers.get("Access-Control-Allow-Origin")
            acac = cors_response.headers.get("Access-Control-Allow-Credentials")
            results["cors"] = {
                "access_control_allow_origin": acao,
                "access_control_allow_credentials": acac,
                "reflects_test_origin": acao == "https://security-check.example.invalid",
                "wildcard_origin": acao == "*",
                "note": (
                    "Basic observation only. Review CORS policy manually; wildcard origins "
                    "or reflected origins can be risky depending on credentials and data."
                )
            }
        except requests.RequestException as exc:
            results["errors"].append(f"CORS check failed: {exc}")

    except requests.RequestException as exc:
        results["errors"].append(f"Request failed: {exc}")

    return results

def print_report(results):
    print("=" * 68)
    print("AUTOMATED WEB SECURITY CHECK REPORT")
    print("=" * 68)
    print("Target:", results["target"])
    print("Checked at:", results["checked_at_utc"])
    print("HTTP status:", results["http_status"])
    print("Final URL:", results["final_url"])
    print("\nSecurity response headers:")
    for name, item in results["security_headers"].items():
        status = "PRESENT" if item["present"] else "MISSING"
        print(f"- {name}: {status}")
        if item["value"]:
            print("  Value:", item["value"])
        print("  Purpose:", item["purpose"])
    print("\nBasic CORS observation:")
    if results["cors"]:
        print(json.dumps(results["cors"], indent=2))
    else:
        print("No CORS result available.")
    if results["errors"]:
        print("\nErrors:")
        for error in results["errors"]:
            print("-", error)
    print("\nReminder: missing headers are findings to review, not proof of a vulnerability.")

# Run this cell and enter a site you own or have explicit permission to test.
target_url = input("Enter an authorized website URL (e.g., https://example.com): ").strip()
report = check_website(target_url)
print_report(report)

# Optional: save the results as JSON for your project evidence.
output_file = "scan_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"\nResults saved to {output_file}")
